# Week 6 skills notebook: rates per person and base rates

This notebook belongs to the skills track. It uses made-up towns, villages, tests and payments, so that the skills stay separate from this week's project.

You will turn counts into rates, catch an AI that divided by the wrong population, see why small places sit at both ends of a ranking, translate a doubled risk into people, and work out what a positive test means.

**How to use it:** run each cell in order. Before you run a cell that prints a ranking, guess who comes first and write your guess down.

## 1. A count, a base and a period

A count says how many times something happened. A rate divides that count by a base, the number of people who could have been counted, over one period. Without the base, a count cannot tell you where a problem is worse.

The table below holds eight made-up towns. It has two population columns: the census of 2015 and an estimate for 2025. The flu cases were reported in 2025.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(25)
pop_2015 = rng.integers(4_000, 60_000, size=8)          # census count in 2015
growth = rng.uniform(0.9, 1.7, size=8)                   # some towns shrank, some grew fast
pop_2025 = np.round(pop_2015 * growth).astype(int)       # estimated population in 2025
true_share = rng.uniform(0.008, 0.030, size=8)           # share of people who catch flu
cases = rng.binomial(pop_2025, true_share)               # flu cases the clinics reported in 2025

towns = pd.DataFrame({"town": list("ABCDEFGH"), "pop_2015": pop_2015,
                      "pop_2025": pop_2025, "cases_2025": cases})
towns

## 2. Describe the table, and match the years

Answer the four questions from week 1: rows, columns, types and missing values. This week adds a fifth question: **which base belongs to the count?** The cases are from 2025, so the base must be the population of 2025. The cell also shows how much each town changed since the census.

In [ ]:
print("rows and columns:", towns.shape)
print()
print(towns.dtypes)
print()
print("missing values:", int(towns.isna().sum().sum()))
print()
change = (towns["pop_2025"] / towns["pop_2015"] - 1) * 100
print("population change since the 2015 census, in percent:")
print(pd.Series(change.round(1).values, index=towns["town"]).to_string())

## 3. Read code an AI assistant wrote

We asked an AI assistant: *"Rank the towns by their flu rate in 2025, per 1,000 people."* It wrote the cell below. Run it, then decide whether the ranking is right.

In [ ]:
# Written by an AI assistant
towns["ai_rate"] = towns["cases_2025"] / towns["pop_2015"] * 1000
ai_ranking = towns.sort_values("ai_rate", ascending=False)
print(ai_ranking[["town", "ai_rate"]].round(2).to_string(index=False))
ai_top = ai_ranking.iloc[0]
print("The AI's highest rate: town", ai_top["town"], "with", round(ai_top["ai_rate"], 2), "per 1,000")

The code runs, and the ranking is still wrong. The assistant divided the cases of 2025 by the census of 2015. A town that grew since 2015 now has more people than its old base, so its rate comes out too high. The towns that grew fastest climb the AI's ranking because they grew, not because more of their people caught flu.

The next cell divides by the population of 2025. Compare the two rankings.

In [ ]:
towns["rate"] = towns["cases_2025"] / towns["pop_2025"] * 1000      # cases of 2025 over people of 2025
ranking = towns.sort_values("rate", ascending=False).reset_index(drop=True)
print(ranking[["town", "rate"]].round(2).to_string(index=False))
top = ranking.iloc[0]
print("Highest rate with the 2025 base: town", top["town"], "with", round(top["rate"], 2), "per 1,000")
place = int(ranking.index[ranking["town"] == ai_top["town"]][0]) + 1
print("Town", ai_top["town"], "with the 2025 base:", round(ranking.loc[place - 1, "rate"], 2), "per 1,000, in place", place, "of 8")
print("Town", ai_top["town"], "grew by", round(change[towns["town"] == ai_top["town"]].iloc[0], 1), "percent since the census")

# CHECK: every rate must use the population of the same year as the cases
assert np.allclose(towns["rate"], towns["cases_2025"] / towns["pop_2025"] * 1000)
assert top["town"] != ai_top["town"]
print("CHECK passed")

## 4. Counts and rates rank differently

Now compare the ranking by number of cases with the ranking by rate. A large town can have the most cases and the lowest rate at the same time. The chart puts the two side by side, with the towns in the same order in both panels.

In [ ]:
by_count = towns.sort_values("cases_2025", ascending=False)["town"].tolist()
by_rate = towns.sort_values("rate", ascending=False)["town"].tolist()
print("ranked by cases:", by_count)
print("ranked by rate: ", by_rate)
most = towns.loc[towns["cases_2025"].idxmax()]
print("Most cases: town", most["town"], "with", int(most["cases_2025"]), "cases")
print("Town", most["town"], "by rate: place", by_rate.index(most["town"]) + 1, "of 8, with", round(most["rate"], 2), "per 1,000")

In [ ]:
import matplotlib.pyplot as plt

order = towns.sort_values("cases_2025")
fig, (left, right) = plt.subplots(1, 2, figsize=(9, 3.5), sharey=True)
left.barh(order["town"], order["cases_2025"])
left.set_title("Flu cases, 2025")
left.set_xlabel("cases")
left.set_ylabel("town")
right.barh(order["town"], order["rate"])
right.set_title("Flu cases per 1,000 people, 2025")
right.set_xlabel("cases per 1,000 people")
plt.tight_layout()
plt.show()

## 5. Your turn: litres per person per day

A rate per person often needs a change of units first. The table below gives the water that five made-up towns received in one year, in cubic metres. One cubic metre is 1,000 litres, and a year has 365 days.

Ask your AI assistant for code that adds a column `litres_per_person_per_day` and stores in `highest_use` the town with the highest value. Paste its code into the cell, run it, and then run the CHECK cell. Before you trust the result, work out one town by hand.

In [ ]:
rng = np.random.default_rng(66)
water = pd.DataFrame({"town": list("PQRST"), "people": rng.integers(2_000, 40_000, size=5)})
water["m3_per_year"] = np.round(water["people"] * rng.uniform(40, 90, size=5), -2)   # cubic metres in one year
water

In [ ]:
# your turn 1: add the column litres_per_person_per_day and store in highest_use the town with the highest value
highest_use = None

In [ ]:
# CHECK
assert highest_use is not None, "highest_use is still empty"
expected = water["m3_per_year"] * 1000 / water["people"] / 365
assert np.allclose(water["litres_per_person_per_day"], expected), "check the units: litres, per person, per day"
assert highest_use == water.loc[expected.idxmax(), "town"]
print("CHECK passed: town", highest_use, "uses", round(expected.max(), 1), "litres per person per day")

## 6. Small bases make extreme rates

The next cell makes 300 villages of very different sizes. Every village has the same true rate of 4 cases per 1,000 people a year. Chance alone decides how many cases each village records. Before you run it, guess which villages will show the highest rates.

In [ ]:
rng = np.random.default_rng(61)
people = np.round(np.exp(rng.uniform(np.log(200), np.log(20_000), size=300))).astype(int)
cases = rng.binomial(people, 0.004)                      # the same true rate everywhere: 4 per 1,000
villages = pd.DataFrame({"people": people, "cases": cases})
villages["rate"] = villages["cases"] / villages["people"] * 1000

top10 = villages.sort_values("rate", ascending=False).head(10)
print(top10.round(2).to_string(index=False))
print()
small = villages[villages["people"] < 1000]
large = villages[villages["people"] >= 10_000]
print("Villages under 1,000 people:", len(small), "with rates from", round(small["rate"].min(), 2), "to", round(small["rate"].max(), 2), "per 1,000")
print("Villages of 10,000 people or more:", len(large), "with rates from", round(large["rate"].min(), 2), "to", round(large["rate"].max(), 2), "per 1,000")
print("Of the 10 highest rates,", int((top10["people"] < 1000).sum()), "are in villages under 1,000 people")
print("One extra case adds", 1 / 200 * 1000, "per 1,000 in a village of 200 people")
print("One extra case adds", 1 / 20_000 * 1000, "per 1,000 in a village of 20,000 people")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.scatter(villages["people"], villages["rate"], s=12)
ax.axhline(4, color="grey", linewidth=1)
ax.set_xscale("log")
ax.set_title("300 villages with the same true rate of 4 per 1,000 (grey line)")
ax.set_xlabel("people in the village (log scale)")
ax.set_ylabel("cases per 1,000 people")
plt.tight_layout()
plt.show()

The highest and the lowest rates both belong to small villages. One extra case moves a small village's rate a long way and a large village's rate hardly at all, so a ranking of rates puts small places at both ends even when nothing differs between them.

When you report a rate, report its count and its base beside it, so the reader can see how much weight the rate can carry.

## 7. A doubled risk, in people

A headline says a habit doubles the risk of an illness. That sentence gives the relative risk and hides the base. The cell below takes two cases: a rare illness that goes from 3 in 10,000 people to 6 in 10,000, and a common one that goes from 20 in 100 to 40 in 100. Both risks double. Count the extra people in each case.

In [ ]:
cases_risk = [("rare illness", 3 / 10_000, 6 / 10_000), ("common illness", 20 / 100, 40 / 100)]
for name, before, after in cases_risk:
    print(f"{name}: relative risk {after / before:.1f}")
    print(f"  per 10,000 people: {before * 10_000:.0f} before and {after * 10_000:.0f} after")
    print(f"  extra cases per 10,000 people: {(after - before) * 10_000:.0f}")

The relative risk is the same in both cases, and the number of extra people is very different. To write the honest version of the headline, give the count per 1,000 or per 10,000 people before and after.

## 8. How common was it before you looked

A test for an illness catches 90% of the people who have it and wrongly flags 5% of the people who do not. A person tests positive. How likely is it that they have the illness? The answer depends on the base rate: how common the illness was before anyone was tested.

The function below counts what happens when 10,000 people take the test. Counting people, instead of multiplying percentages, is called working with natural frequencies. The cell runs the same test at three base rates.

In [ ]:
def screen(base_rate, catches, false_alarm, people=10_000):
    # natural frequencies: count what happens when this many people are tested
    real = people * base_rate
    true_pos = real * catches
    false_pos = (people - real) * false_alarm
    share_real = true_pos / (true_pos + false_pos)
    print(f"base rate {base_rate:.1%}: {real:.0f} real cases, {true_pos:.0f} true positives, "
          f"{false_pos:.0f} false positives, {share_real:.1%} of positives are real")
    return share_real

for b in [0.01, 0.10, 0.30]:
    screen(b, catches=0.90, false_alarm=0.05)

When 1 person in 100 has the illness, most positive results are false alarms. The test is the same in all three rows, and only the base rate changes. The next cell checks the counting with a simulation of 100,000 people.

In [ ]:
rng = np.random.default_rng(3)
n = 100_000
ill = rng.random(n) < 0.01
positive = np.where(ill, rng.random(n) < 0.90, rng.random(n) < 0.05)
print("simulated people:", n)
print("positive results:", int(positive.sum()))
print("share of positives who are ill:", round(ill[positive].mean() * 100, 1), "percent")

## 9. Your turn: fraud alerts

The same arithmetic applies wherever a rare event is screened for, such as fraud, security threats or hiring. A bank's alert catches 90% of fraudulent payments and wrongly flags 1% of honest ones. Two payments in 1,000 are fraud.

First ask your AI assistant: *"What share of flagged payments are fraud?"* Write down its answer. Then store the share yourself in `share_fraud`, as a number between 0 and 1, with `screen` or with your own code, and run the CHECK cell.

In [ ]:
# your turn 2: store in share_fraud the share of flagged payments that are fraud
share_fraud = None

In [ ]:
# CHECK
assert share_fraud is not None, "share_fraud is still empty"
expected = 0.002 * 0.90 / (0.002 * 0.90 + 0.998 * 0.01)
assert abs(share_fraud - expected) < 0.001, "count the honest payments that are flagged, too"
print("CHECK passed:", round(expected * 100, 1), "percent of flagged payments are fraud")

## 10. Write it for a reader

In a text cell below, write two short paragraphs to a friend whose test from section 8 came back positive, at a base rate of 1 in 100. Use counts of people, not percentages. End with what the friend should do next.